# Cholesky Multivariate Stochastic Volatility (ChMSV) Project

**Reference Paper:** Szczepocki, P. (2023). *Estimation of the Cholesky Multivariate Stochastic Volatility Model Using Iterated Filtering.* Econometrics. Ekonometria. Advances in Applied Data Analysis, 27(4), 44–58.  
**Course:** Sequential Monte Carlo (`particles`) — Prof. Nicolas Chopin  
**Final Deadline:** January 10

---

## 1. Data & Base ChMSV Model Setup (Role 1 — Gabriel)

### 1.1 Mathematical Formulation of the ChMSV State-Space Model
For a $p$-dimensional vector of asset log-returns (multiplied by 100) $\mathbb{y}_t = (y_{1,t}, \dots, y_{p,t})'$, the Cholesky Multivariate Stochastic Volatility (ChMSV) model (Equations 1–9 in Szczepocki, 2023) is defined by:

$$\begin{aligned}
\mathbb{y}_t &= \mathbb{L}_t \mathbb{V}_t^{1/2} \mathbb{e}_t, \quad \mathbb{e}_t \sim \text{iid } \mathcal{N}_p(0, \mathbb{I}_p), \\
\mathbb{L}_t &= \begin{pmatrix} 1 & 0 & \cdots & 0 \\ q_{21,t} & 1 & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ q_{p1,t} & q_{p2,t} & \cdots & 1 \end{pmatrix}, \quad
\mathbb{V}_t^{1/2} = \text{diag}\!\left(\exp\!\left(\frac{h_{1,t}}{2}\right), \dots, \exp\!\left(\frac{h_{p,t}}{2}\right)\right), \\
h_{i,t} &= \mu_i + \phi_i (h_{i,t-1} - \mu_i) + \eta_{i,t}, \quad \eta_{i,t} \sim \text{iid } \mathcal{N}(0, \sigma_{\eta_i}^2), \quad i = 1, \dots, p, \\
q_{ij,t} &= \mu_{ij} + \phi_{ij} (q_{ij,t-1} - \mu_{ij}) + \upsilon_{ij,t}, \quad \upsilon_{ij,t} \sim \text{iid } \mathcal{N}(0, \sigma_{\psi_{ij}}^2), \quad 1 \le j < i \le p.
\end{aligned}$$

* **Latent State Vector ($d_x = p(p+1)/2$):** $X_t = (h_{1,t}, \dots, h_{p,t}, q_{21,t}, \dots, q_{p,p-1,t})'$ ($d_x = 10$ states for $p=4$; $d_x = 3$ states for $p=2$).
* **Initial State $X_0$:** Point-mass (Dirac) at $(\mu_1, \dots, \mu_p, \mu_{21}, \dots, \mu_{p,p-1})'$ (Section 3.1, p. 48).
* **Time-Varying Covariance $\Sigma_t = \mathbb{L}_t \mathbb{V}_t \mathbb{L}_t^T$ (Equations 4 & 5):**
  $$\Sigma_{t,ii} = \sum_{k=1}^i q_{ik,t}^2 \exp(h_{k,t}) \quad (q_{ii,t}=1), \qquad \Sigma_{t,ij} = \sum_{k=1}^j q_{ik,t} q_{jk,t} \exp(h_{k,t}) \quad (i > j).$$
* **Return Volatilities & Dynamic Correlations (Equations 6 & 7):**
  $$\text{Stdev}(y_{i,t} \mid \mathcal{F}_{t-1}) = \sqrt{\Sigma_{t,ii}}, \qquad \text{Cor}(y_{i,t}, y_{j,t} \mid \mathcal{F}_{t-1}) = \frac{\Sigma_{t,ij}}{\sqrt{\Sigma_{t,ii}\Sigma_{t,jj}}}.$$

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from particles import state_space_models as ssm

from src.base_model import (
    ChMSV,
    ChMSV4Asset,
    ChMSV2Asset,
    TABLE2_PARAMS_4ASSET,
    TABLE2_PARAMS_2ASSET,
    states_to_cholesky_factors,
    states_to_covariance,
    states_to_volatility_and_correlation,
    load_real_dataset,
    load_simulated_chmsv_dataset,
    compute_table1_summary_stats,
    plot_figure1_real_data,
    plot_simulated_datasets_overview,
)

### 1.2 Real 4-Asset Dataset (SPX, Gold, Bitcoin, Ethereum: June 4, 2018 – June 4, 2023)
We load the cleaned real 4-asset dataset (`data/processed/real_4assets_log_returns.csv`, $T = 1258$ daily percentage log-returns $100 \log(P_t/P_{t-1})$) and verify the summary statistics against **Table 1** and **Figure 1** of Szczepocki (2023).

In [ ]:
real4 = load_real_dataset(p=4)
print(f"Loaded Real 4-Asset Dataset: y_obs shape = {real4['y_obs'].shape}")
display(compute_table1_summary_stats(real4["df"]))

fig1 = plot_figure1_real_data(real4, save_path=None)
plt.show()

### 1.3 Simulated Test Datasets (Fast 2-Asset & Full 4-Asset using Table 2 Parameters)
We load both simulated datasets generated from `ChMSV` using the paper's Table 2 parameters, and demonstrate `states_to_volatility_and_correlation` (Equations 6 & 7):
* **Fast 2-Asset Dataset (`p = 2`):** SPX & Gold, 3 hidden states ($h_1, h_2, q_{21}$), 9 parameters, $T = 500$.
* **Full 4-Asset Dataset (`p = 4`):** SPX, Gold, BTC, ETH, 10 hidden states, 30 parameters, $T = 1258$.

In [ ]:
sim2 = load_simulated_chmsv_dataset(p=2)
sim4 = load_simulated_chmsv_dataset(p=4)

print(f"2-Asset Simulated: y_obs={sim2['y_obs'].shape}, x_true={sim2['x_true'].shape}")
print(f"4-Asset Simulated: y_obs={sim4['y_obs'].shape}, x_true={sim4['x_true'].shape}")

# Example usage of state-to-volatility/correlation transformation (Equations 6 & 7)
transformed_4 = states_to_volatility_and_correlation(sim4["x_true"], p=4)
print("Transformed stdev shape:", transformed_4["stdev"].shape, "| corr_pairs shape:", transformed_4["corr_pairs"].shape)

fig_sim = plot_simulated_datasets_overview(sim4, sim2, save_path=None)
plt.show()

---

## 2. Task 1 – Bootstrap Particle Filter & Dynamic Correlation Tracking (Role 2)

*Placeholder section for Role 2:*
* *M1: Bootstrap Particle Filter pipeline to extract filtered volatilities, dynamic correlations (Equations 6 & 7), and log-likelihood evaluations.*
* *M2: 10-run variability benchmark on simulated data across particle counts $N$ + state-recovery plots.*
* *M3: Real 4-asset dataset run using Table 2 parameters (and Role 3 estimates) reproducing Figures 2, 3, and 4.*

In [ ]:
# Role 2 (Task 1): Add Bootstrap Particle Filter & Dynamic Correlation Tracking code here


---

## 3. Task 2 & 3 – Iterated Filtering (IF2) Implementation (Role 3)

*Placeholder section for Role 3:*
* *M1: IF2 algorithm (Ionides et al., 2015) with random-walk parameter perturbations and geometric cooling ($\alpha = 0.5$).* 
* *M2: 10+ run parameter recovery benchmark on simulated data, convergence trajectories, and ME/RMSE table (Table 3).* 
* *M3: Real 4-asset IF2 estimation and comparison against Table 2.*

In [ ]:
# Role 3 (Task 2 & 3): Add Iterated Filtering (IF2) implementation and convergence plots here


---

## 4. Task 4 – Competing Parameter Estimation Method: PMCMC / $\text{SMC}^2$ (Role 4)

*Placeholder section for Role 4:*
* *M1: Prior distributions (`StructDist`) and PMMH / Particle Gibbs / $\text{SMC}^2$ setup in `particles`.* 
* *M2: Multi-chain parameter recovery on simulated data (starting with 2-asset subset `ChMSV2Asset`).* 
* *M3: Real dataset estimation and head-to-head comparison against Role 3's IF2 results.*

In [ ]:
# Role 4 (Task 4): Add PMCMC / SMC^2 parameter estimation code and comparison plots here
